# CONDACT quick-start tutorial (Google Colab)

**CONDitional ACTivity, a kinetic framework for mapping allosteric communication**

Loverde Laboratory, CUNY. Code by Augustine C. Onyema, assisted by Chukwuebuka Dikeocha.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CUNY-CSI-Loverde-Laboratory/condact-colab-tutorial/blob/main/CONDACT_Colab_Tutorial.ipynb)

This tutorial is written for someone who is new to CONDACT and fairly new to analysing molecular dynamics (MD) simulations. You do not need to install anything on your own computer. Everything runs in your browser and takes about five minutes.

## Before you start

- **How to run a cell:** click on a grey code cell and press **Shift + Enter**, or click the ▶ button on its left. Run the cells in order, from top to bottom.
- **To run everything at once:** use the menu **Runtime → Run all**.
- **If something breaks:** use **Runtime → Restart session and run all** to start again from a clean state.
- You do not need a GPU.
- You can change anything you like. Your edits are not saved to the original unless you choose **File → Save a copy in Drive**.

## What question does CONDACT answer?

Proteins and nucleic acids send signals from one site to another (*allostery*). CONDACT looks for this in a simulation by asking a simple timing question:

> **When residue Y changes its conformation, does residue X tend to change soon afterwards?**

If yes, the two residues are *kinetically coupled*, and Y may be passing information to X.

## Four terms you need

| Term | Plain meaning |
|---|---|
| **State** | Which of a few preferred positions a side chain is in. We describe a side chain by one angle, its first side-chain dihedral **χ1** ("chi-one"), which usually sits near one of three values. |
| **Transition** | The moment a residue jumps from one state to another (a side-chain "flip"). |
| **Persistence time** | How long a residue typically waits between its own transitions. |
| **Conditional activity, A[X][Y]** | Compares how long X waits to flip *after Y has flipped* with how long X normally waits. |

The formula is

$$A[X][Y] = -\ln\left(\frac{\text{time X waits after Y flips}}{\text{time X normally waits}}\right)$$

and you read the number like this:

| Value | Meaning |
|---|---|
| `A[X][Y] > 0` | X flips **sooner** than usual after Y flips: Y promotes X |
| `A[X][Y] ≈ 0` | X does not care what Y does |
| `A[X][Y] < 0` | X flips **later** than usual after Y flips: Y suppresses X |
| `A[X][X]` | *Dynamical memory*: positive means X's own flips come in bursts |

The order matters: `A[X][Y]` and `A[Y][X]` are usually different numbers. That is what lets CONDACT say which way the signal goes.

## What you will do

1. Install CONDACT
2. Load a trajectory
3. Look at the angles and decide where one state ends and the next begins
4. Compute the conditional activity
5. Read the results
6. Check which results are statistically significant
7. Going further (optional): connected domains and communication routes
8. Use your own trajectory

## 1. Install CONDACT

Run the cell below. It downloads CONDACT and MDAnalysis (the library used to read trajectories). It takes about a minute, and it is normal for it to print nothing.

In [ ]:
!pip install -q git+https://github.com/CUNY-CSI-Loverde-Laboratory/conditional_activity.git

Now load the tools. You should see the version numbers printed at the end. (CONDACT also prints a line about "Total runtime"; you can ignore it.)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda

import conditional_activity
from conditional_activity.analysis.condact import CONDACT
from conditional_activity.analysis.fdr import CONDACT_FDR
from conditional_activity.analysis.trace_route import CONDACT_CommPath

print("CONDACT", conditional_activity.__version__, "| MDAnalysis", mda.__version__)

## 2. Load a trajectory

A simulation is stored as two files:

- a **topology** file, which lists the atoms and residues, and
- a **trajectory** file, which stores the atom positions at each saved time point (a *frame*).

CONDACT comes with a tiny example so you can practise: three amino-acid residues (GLN 1, ASN 2 and ASP 3) taken from a larger simulation, 30 nanoseconds long, with a frame saved every 10 picoseconds.

The three settings in capital letters are the ones you will change when you use your own data.

In [ ]:
# Example files that were installed with the package
example_dir = os.path.join(os.path.dirname(conditional_activity.__file__), "analysis")
topology_file = os.path.join(example_dir, "pytest_LYS.prmtop")
trajectory_file = os.path.join(example_dir, "pytest_LYS.xtc")

SELECTED_RESID = "1-3"   # which residues to analyse, e.g. "1-3" or "5 9 12"
SAVING_FREQUENCY = 10    # time between saved frames, in picoseconds (ps)

# A "Universe" is MDAnalysis's name for a loaded simulation
u = mda.Universe(topology_file, trajectory_file)
TOTAL_TIME = len(u.trajectory) * SAVING_FREQUENCY   # length of the simulation in ps

print(f"{u.atoms.n_atoms} atoms, {len(u.trajectory)} frames, {TOTAL_TIME/1000:.0f} ns")
print("Residues:", ", ".join(f"{r.resname} {r.resid}" for r in u.select_atoms(f"resid {SELECTED_RESID}").residues))

**Check:** you should see `43 atoms, 3000 frames, 30 ns` and the three residue names.

`SAVING_FREQUENCY` is how often frames were written to the trajectory file. It is *not* the MD time step (which is usually 2 femtoseconds). If you are unsure, look for the output frequency in your simulation input file.

## 3. Look at the angles and choose the state boundaries

First we measure χ1 for each residue in every frame. χ1 is the dihedral angle defined by four atoms: N, CA, CB and the first side-chain atom after CB (CG for most residues). Alanine and glycine have no such atom, so CONDACT skips them.

The cell below only measures the angles. You do not need to follow every line.

In [ ]:
def chi1_atoms(residue):
    # The four atoms that define chi1, in order
    atoms = residue.atoms
    return (atoms.select_atoms("name N") + atoms.select_atoms("name CA") + atoms.select_atoms("name CB")
            + atoms.select_atoms("name CG or name OG1 or name CG1 or name OG or name SG"))

selection = u.select_atoms(f"resid {SELECTED_RESID} and not (resname ALA or resname GLY)")
groups = {f"{res.resname} {res.resid}": chi1_atoms(res) for res in selection.residues}

angles = {name: [] for name in groups}
for frame in u.trajectory:
    for name, atom_group in groups.items():
        angle = atom_group.dihedral.value()   # between -180 and 180 degrees
        angles[name].append(angle % 360)      # shift to 0-360 degrees, as CONDACT does

angles = pd.DataFrame(angles)
angles.index = angles.index * SAVING_FREQUENCY / 1000   # time in ns
angles.index.name = "time (ns)"
angles.head()

### What does a transition look like?

Here is χ1 for one residue over time. The angle hovers around one value, then suddenly jumps to another. Each jump across a dashed line is one **transition**.

In [ ]:
residue_to_plot = "GLN 1"

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(angles.index, angles[residue_to_plot], ".", markersize=2)
for boundary in (120, 240):
    ax.axhline(boundary, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Time (ns)")
ax.set_ylabel("χ1 (degrees)")
ax.set_ylim(0, 360)
ax.set_yticks([0, 120, 240, 360])
ax.set_title(residue_to_plot)
plt.tight_layout()
plt.show()

### Where are the states?

A histogram of all the angles shows which values are preferred. Each **peak** is a state, and each **valley** between peaks is a boundary.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 360, 73)
for name in angles.columns:
    ax.hist(angles[name], bins=bins, density=True, histtype="step", linewidth=1.8, label=name)
for boundary in (120, 240):
    ax.axvline(boundary, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("χ1 (degrees)")
ax.set_ylabel("Probability density")
ax.set_xlim(0, 360)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**What you should see:** three peaks, near 60°, 190° and 300°, with empty valleys near **120°** and **240°** (the dashed lines).

So for this system we will tell CONDACT:

- there are **3** states,
- the boundaries between them are at **120°** and **240°**,
- and we will name the states **X**, **Y** and **Z** (any letters will do).

These happen to be CONDACT's default values for proteins, but always make this plot for your own system to check that they fit.

## 4. Compute the conditional activity

Now we hand everything to CONDACT. The `..._nucleic` settings are only used when you select DNA or RNA residues, so here they are left at their defaults.

In [ ]:
study = CONDACT(
    universe=u,
    selected_resid=SELECTED_RESID,
    No_of_peaks_protein=3,               # number of states (peaks) in the histogram
    peak_boundaries_protein=[120, 240],  # the valleys between them
    states_protein="XYZ",                # one letter per state
    No_of_peaks_nucleic=3,               # the next three lines are only used for DNA/RNA
    peak_boundaries_nucleic=[140, 330],
    states_nucleic="SAS",
    saving_frequency=SAVING_FREQUENCY,
    keep_negative=False,                 # False = use angles from 0 to 360 degrees
)

conditional_activity_list, persistence_times, exchange_times = study.mean_conditional_activity()

# Put the result in a table with residue names
CA = np.array(conditional_activity_list, dtype=float)
_, residue_names, residue_type = study.residues()
labels = [f"{name} {res.resid}" for name, res in zip(residue_names, residue_type)]
print("\nDone.")

**About the messages above.** Lines saying a residue "had statistically fewer transitions" are not errors. CONDACT only trusts a residue that flipped **at least 10 times**. In this short example ASN 2 flipped only 3 times, so every result involving ASN 2 is reported as `NaN` ("not a number", meaning "not enough data"). This is why real analyses need long simulations or a simulation with residues adopting multiple transitions.

CONDACT also saved its results as files. Click the folder icon in the left sidebar to see them; you can download any of them.

| File | Contents |
|---|---|
| `Dihedral_Angle.csv` | the angle of every selected residue in every frame |
| `transition_times.pkl` | the times at which each residue flipped |
| `wait_times.pkl` | the waiting times between flips |
| `Conditional_Activity_matrix.npy` | the conditional activity table |
| `Dynamic_Memory.npy` | the dynamical memory of each residue |

## 5. Read the results

### One line per residue

In [ ]:
summary = pd.DataFrame({
    "number of transitions": [len(study.transition_times[l]) for l in labels],
    "persistence time (ps)": [persistence_times[l] for l in labels],
    "dynamical memory A[X][X]": np.diag(CA),
}, index=labels)
summary.loc[summary["number of transitions"] < 10, "persistence time (ps)"] = np.nan   # too few flips to estimate
summary.round(2)

**How to read this table**

- GLN 1 flipped often (64 times), ASP 3 flipped 12 times, and ASN 2 flipped only 3 times, which is too few to analyse (so its other columns are `NaN`).
- The **persistence time** is in picoseconds: GLN 1 typically stays put for about 1 ns between flips, ASP 3 for about 4 ns.
- **Dynamical memory** is positive for GLN 1 and ASP 3 (about 1.1 and 1.6). That means their flips are not evenly spread in time: they come in bursts, with quiet periods in between.

### The conditional activity table

Each cell answers: *after the residue in the **column** flips, does the residue in the **row** flip sooner than usual?*

In [ ]:
display(pd.DataFrame(CA, index=labels, columns=labels).round(2))

fig, ax = plt.subplots(figsize=(4.8, 4))
image = ax.imshow(CA, cmap="hot_r", vmin=0, origin="upper")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Y: the residue that flips first")
ax.set_ylabel("X: the residue that responds")
fig.colorbar(image, label="Conditional activity A[X][Y]")
plt.tight_layout()
plt.show()

**How to read this table**

- Row GLN 1, column ASP 3 is about **2.2**, a large positive number: after ASP 3 flips, GLN 1 flips much sooner than it normally would. Signal direction: **ASP 3 → GLN 1**.
- Row ASP 3, column GLN 1 is only about **0.2**: a flip of GLN 1 has little effect on ASP 3. The coupling is mostly one-way.
- The white cells are `NaN` (ASN 2 did not flip often enough).
- The diagonal is the dynamical memory from the table above.

A large number is suggestive, but it could still be a coincidence. The next section tests that.

## 6. Check which results are statistically significant

**The idea.** If ASP 3 really influences GLN 1, the *timing* of their flips must line up. So we deliberately break the timing: slide all of ASP 3's flip times by a random amount and recompute the conditional activity. Doing this many times shows what values arise by chance alone.

- The **p-value** is the fraction of these shuffled runs that scored at least as high as the real result. Small p-value = unlikely to be chance.
- Because we test many residue pairs at once, some will look significant by luck. The **q-value** is the p-value corrected for this (a *false discovery rate*, or FDR, correction). **Use the q-value to decide**; below 0.05 is the usual cut-off.

`n_perm` is the number of shuffles. 2,000 keeps this example quick. Use 10,000 or more for results you plan to publish.

After applying FDR corrections only CA values which are statistically significant should be used for downstream analysis and results interpretation.

In [ ]:
fdr = CONDACT_FDR(
    transition_times_file="transition_times.pkl",
    CA_matrix_file="Conditional_Activity_matrix.npy",
    total_time=TOTAL_TIME,   # length of the simulation in ps
    n_perm=2000,             # number of random shuffles
    q=0.05,                  # significance cut-off for the q-value
    CA_cutoff=0.0,           # only test pairs with conditional activity above this
    random_seed=0,           # makes the result reproducible
)
pvals, qvals, significant = fdr.run()

results = pd.read_csv("CA_all_tested_edges.csv")
results = results.rename(columns={"Residue_i": "X (responds)", "Residue_j": "Y (flips first)"})
results[["Y (flips first)", "X (responds)", "Conditional_Activity", "p_value", "q_value", "Significant_FDR"]].round(3)

In [ ]:
### Significant qvalues entries
qvals

**Apply FDR correction to the conditional activity matrix**

- Retain only entries with a q-value ≤ 0.05 (statistically significant after multiple-testing correction). All other entries — including untested pairs (NaN q-values such as self-interactions) and pairs that failed FDR (q > 0.05) are set to NaN, ensuring that only genuine allosteric couplings propagate into downstream analysis.

In [ ]:
### CA cleaned up to only include significant q-values entries
CA_filtered = CA.copy()
CA_filtered = np.where(qvals <= 0.05, CA_filtered, np.nan)
CA_filtered

**How to read this table**

- **ASP 3 → GLN 1** has a q-value below 0.05 and `Significant_FDR = True`. This coupling is unlikely to be a coincidence.
- **GLN 1 → ASP 3** has a large q-value, so there is no evidence for coupling in that direction.

The full table is saved as `CA_all_tested_edges.csv`, and the significant rows alone as `CA_significant_edges.csv`.

**That is the core workflow.** Sections 1 to 6 are everything you need for a first analysis.

### Try it yourself

1. In section 3, change `residue_to_plot` to `"ASP 3"` and re-run that cell. Can you count roughly how many times it crosses a dashed line? Compare with the "number of transitions" in section 5.
2. In section 4, change the boundaries to `[130, 230]` and re-run sections 4 and 5. The numbers barely change, because both boundaries still sit in empty valleys. Now try `[100, 260]`. The ASP 3 → GLN 1 value drops from about 2.2 to about 0.7. Look at the histogram to see why: 260° cuts into the edge of the peak near 300°, so ordinary wobbling inside one state gets counted as transitions. This is why you choose boundaries from the histogram. (Set them back to `[120, 240]` afterwards.)
3. In section 6, change `random_seed` to another number and re-run. The p-values move a little. Now set `n_perm=10000`: they move less. Why?

## 7. Going further (optional)

Skip this section on a first read. These two analyses become useful once you are working with a whole protein or complex rather than three residues.

### Dynamically connected domains

Residues that are all coupled to one another form a group that moves "in step". The principal eigenvector of the conditional activity table picks out the largest such group: residues with large values belong to it, residues near zero do not.

In [ ]:
CA_filled = np.nan_to_num(CA_filtered, nan=0.0)           # treat "not enough data" as no coupling
CA_symmetric = 0.5 * (CA_filled + CA_filled.T)   # average the two directions
eigenvalues, eigenvectors = np.linalg.eigh(CA_symmetric)
principal = np.abs(eigenvectors[:, np.argmax(eigenvalues)])

pd.Series(principal, index=labels, name="weight in the main coupled group").round(2)

Here GLN 1 and ASP 3 form the group and ASN 2 is left out, as expected.

### Communication routes

In a large system, a signal can travel through a chain of residues: A → B → C. `CONDACT_CommPath` finds such chains. It links two residues when (1) their conditional activity is positive and (2) they are close together in space, then follows the strongest links.

For that it needs the average distance between each pair of residues:

In [ ]:
from MDAnalysis.analysis import distances

residues_sel = u.select_atoms(f"resid {SELECTED_RESID} and not (resname ALA or resname GLY)")
dist = np.zeros((len(labels), len(labels)))
for frame in u.trajectory:
    centers = residues_sel.center_of_geometry(compound="residues")
    dist += distances.distance_array(centers, centers, box=u.dimensions)
dist /= len(u.trajectory)

pd.DataFrame(dist, index=labels, columns=labels).round(1)   # in Angstrom

In [ ]:
comm = CONDACT_CommPath(CA_filled, dist, labels=labels)

# cutoff = how close two residues must be (in Angstrom) to pass a signal directly.
# These three example residues are far apart, so we use a very loose 30 A here.
# For a real system use a contact distance, for example 5 to 8 A.
paths = comm.enumerate_paths(start=None, start_mode="global_max",
                             cutoff=30.0, min_ca=0.0, max_depth=5, top_k=10)
comm.print_paths(paths, top=10)

With three residues there is only one route, ASP 3 → GLN 1, which matches what we found in section 6. To ask about a particular start and end point in your own system:

```python
paths = comm.trace_greedy(start="ASP 3", target="GLN 1", cutoff=30.0, min_ca=0.0, branch_k=10)
comm.print_paths(paths)
```

## 8. Use your own trajectory

**Step 1. Get your files into Colab.** Run one of these in a new code cell.

Upload from your computer (best for small files):

```python
from google.colab import files
uploaded = files.upload()
```

Or connect your Google Drive (best for large files):

```python
from google.colab import drive
drive.mount("/content/drive")      # your files are then under /content/drive/MyDrive/
```

**Step 2. Edit section 2.** Replace the two file names and the two settings, for example:

```python
topology_file = "my_protein.prmtop"
trajectory_file = "my_protein.xtc"
SELECTED_RESID = "20-60"
SAVING_FREQUENCY = 100
```

**Step 3. Re-run from section 2 downwards.** Look at the histogram in section 3 and, if your valleys are in different places, change the boundaries in section 4.

**Advice for a first real analysis**

- **Use a long trajectory.** A residue needs at least 10 flips. If most of your table is `NaN`, the simulation is too short for those residues.
- **Start small.** The calculation time grows with the square of the number of residues. Try one region (a few dozen residues) before a whole complex.
- **Shrink your files first.** Colab has limited memory. Remove water and ions from the trajectory before uploading.
- **DNA and RNA work too.** For nucleotides CONDACT uses the sugar-base dihedral. Make the same histogram to check the `..._nucleic` boundaries.
- **Colab disconnects when idle** and deletes its files. Download the results you want to keep.

## If something goes wrong

| What you see | What to do |
|---|---|
| `ModuleNotFoundError: No module named 'conditional_activity'` | Run the install cell in section 1 first, then the import cell. |
| `NameError: name 'u' is not defined` (or similar) | You skipped a cell. Use **Runtime → Run all**. |
| Almost everything is `NaN` | Too few transitions. Use a longer trajectory, or check your boundaries against the histogram. |
| `ValueError` mentioning peaks and boundaries | The number of boundaries must be one less than the number of states (3 states → 2 boundaries). |
| The session crashes or runs out of memory | Your trajectory is too large for Colab. Strip the solvent, keep fewer frames, or run on your own machine. |

## Citation

If you use CONDACT, please cite:

- Onyema, A. C.; Dikeocha, C.; Patel, R.; Moussa, J.; Loverde, S. M. *Mapping Allosteric Communication in the Nucleosome with Conditional Activity.* J. Chem. Inf. Model. (2026). https://pubs.acs.org/doi/10.1021/acs.jcim.6c00317
- Lin, M. M. *Timing correlations in proteins predict functional modules and dynamic allostery.* J. Am. Chem. Soc. 2016, 138 (15), 5036–5043. https://pubs.acs.org/doi/10.1021/jacs.5b08814
- MDAnalysis: https://www.mdanalysis.org/pages/citations/

Code: https://github.com/CUNY-CSI-Loverde-Laboratory/conditional_activity (GPL v2). The trajectories from the paper are on Zenodo: https://zenodo.org/records/17193099